# Reporte de bajo rendimiento académico

Estudiantes con **Nota final < 3.0** agrupados por programa, semestre, sede y asignatura.

In [46]:
import pandas as pd

notas = pd.read_csv("../data/raw/notas_pivot.csv")
est = pd.read_csv("../data/raw/estudiantes.csv")

df = notas.merge(
    est[["Numero_identificacion", "Nombre_nivel", "Sede", "Nombre_programa_limpio"]],
    left_on="Numero_identificacion_estudiante",
    right_on="Numero_identificacion",
    how="left"
)

df["bajo"] = (df['Nota final'] < 3.0) & (df['Nota final'] > 1)
df["condición_de_alerta"] = (df['Nota final'] <=1) & (df['Nota final'] >=0)

print(f"Total registros: {len(df):,}")
print(f"registros con nota < 3.0: {df['bajo'].sum():,}")
print(f"registos con condición de alerta: {df['condición_de_alerta'].sum():,}")

Total registros: 5,572
registros con nota < 3.0: 294
registos con condición de alerta: 281


In [47]:
import unicodedata
# Clasificacion por area
def _asignar_area(nombre):
    nombre = str(nombre).strip()
    nombre = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode("ascii").upper()
    m = {
        "Logistica y Comercio": [
            "GESTION NAVIERA Y PORTUARIA",
            "INTRODUCCION AL COMERCIO EXTERIOR",
            "INTRODUCCION A LA LOGISTICA",
            "INFRAESTRUCTURA PORTUARIA",
            "LOGISTICA COMERCIAL",
            "LEGISLACION COMERCIAL Y ADUANERA",
        ],
        "Marketing y Ventas": [
            "INTRODUCCION AL MARKETING DIGITAL",
            "GESTION DE PRECIOS Y PERCEPCION DE VALOR",
            "SEO Y SEM POSICIONAMIENTO WEB",
            "ESTRATEGIAS DE CONTENIDO EN REDES SOCIALES",
        ],
        "Turismo": [
            "ORGANIZACION DE SERVICIOS EN AGENCIAS DE VIAJES",
            "ETNOTURISMO Y DESARROLLO COMUNITARIO",
            "SOSTENIBILIDAD Y TURISMO REGENERATIVO",
            "LEGISLACION EN LA INDUSTRIA TURISTICA",
            "ETIQUETA Y PROTOCOLO EN SERVICIOS TURISTICOS",
            "FUNDAMENTOS DE LA OPERACION TURISTICA",
        ],
        "Idiomas": [
            "INGLES I",
            "INGLES II",
        ],
        "Matematicas y Estadistica": [
            "RAZONAMIENTO CUANTITATIVO",
            "PROBABILIDAD Y ESTADISTICA",
            "ESTADISTICA",
            "ALGEBRA Y GEOMETRIA ANALITICA",
            "CALCULO DIFERENCIAL",
        ],
        "Comunicacion": [
            "COMUNICACION ESCRITA",
            "EXPRESION ORAL Y ARGUMENTACION",
            "LECTOESCRITURA Y COMUNICACION TECNICA",
        ],
        "Formacion Integral": [
            "ETICA PROFESIONAL EN LOS NEGOCIOS",
            "COMPETENCIAS HUMANAS Y DESARROLLO INTEGRAL",
            "CATEDRA USM Y CONTEXTO SAMARIO",
            "TALLER DE CREATIVIDAD E INNOVACION",
        ],
        "Tecnologia y Datos": [
            "HERRAMIENTAS TICS",
            "HERRAMIENTAS TECNOLOGICAS APLICADAS I",
            "MINERIA DE DATOS I",
        ],
        "Diseno y Modas": [
            "PATRONAJE Y ESCALADO",
            "TECNICAS DE CONFECCION I",
            "PRODUCCION DE MODAS",
            "HISTORIA DE LA MODA E INDUMENTARIA",
        ],
        "Gastronomia": [
            "CULTURA Y GASTRONOMIA",
            "MIXOLOGIA Y COCTELERIA",
            "GASTRONOMIA II",
            "SST Y BPM",
            "GASTRONOMIA I",
            "BIOQUIMICA",
        ],
        "Administracion y Costos": [
            "ADMINISTRACION DE ORGANIZACIONES",
            "COSTOS Y PRESUPUESTOS",
        ],
    }
    for area, materias in m.items():
        if nombre in materias:
            return area
    return "Otra"

df["Area"] = df["Nombre_asignatura"].apply(_asignar_area)
print(df["Area"].value_counts())

Area
Matematicas y Estadistica    876
Comunicacion                 803
Idiomas                      736
Logistica y Comercio         582
Tecnologia y Datos           575
Turismo                      462
Gastronomia                  438
Formacion Integral           381
Marketing y Ventas           380
Diseno y Modas               272
Administracion y Costos       67
Name: count, dtype: int64


In [48]:
print(df.columns)
df['Nombre_curso'].nunique()
#df['bajo']


Index(['Consecutivo_curso', 'Nombre_curso', 'Codigo_asignatura',
       'Nombre_asignatura', 'Codigo_programa', 'Nombre_programa',
       'Consecutivo_periodo', 'Nombre_periodo', 'Consecutivo_sede_jornada',
       'Nombre_sede_jornada', 'Nombre_completo_docente',
       'Numero_identificacion_docente', 'Codigo_matricula',
       'Numero_identificacion_estudiante',
       'Abreviatura_tipo_identificacion_estudiante',
       'Nombre_completo_estudiante', 'Promedio_evaluacion',
       'Porcentaje_evaluado', 'Cantidad_inasistencia',
       'Porcentaje_inasistencia', 'Estudiante_formalizado', 'Observaciones',
       'Primer Seguimiento', 'Segundo Seguimiento', 'Tercer Seguimiento',
       'Grupo', 'Nota final', 'Numero_identificacion', 'Nombre_nivel', 'Sede',
       'Nombre_programa_limpio', 'bajo', 'condición_de_alerta', 'Area'],
      dtype='str')


185

In [ ]:
def _tabla_agrupada(df, groupby_cols):
    return df.groupby(groupby_cols).agg(
        Total_estudiantes=("bajo", "count"),
        Bajo_rendimiento=("bajo", "sum"),
        Porcentaje_bajo=("bajo", "mean")
    ).reset_index().assign(
        Porcentaje_bajo=lambda x: (x["Porcentaje_bajo"] * 100).round(1)
    ).sort_values("Porcentaje_bajo", ascending=False).reset_index(drop=True)


seguimientos = [
    ("Primer Seguimiento", "Seg 1"),
    ("Segundo Seguimiento", "Seg 2"),
    ("Tercer Seguimiento", "Seg 3"),
    ("Nota final", "Final"),
]

for col_nota, titulo in seguimientos:
    print(f"\n=== BAJO RENDIMIENTO — {titulo} ===\n")

    df["bajo"] = (df[col_nota] < 3.0) & (df[col_nota] > 1)
    df["condicion_de_alerta"] = (df[col_nota] <= 1) & (df[col_nota] >= 0)

    print(f"Registros con bajo rendimiento: {df['bajo'].sum()}")
    print(f"Registros en condición de alerta: {df['condicion_de_alerta'].sum()}")

    tabla_curso = _tabla_agrupada(df, ["Nombre_curso", "Sede", "Nombre_programa_limpio", "Grupo", "Area"])
    print(f"\nPor curso (solo con bajo rendimiento > 0):")
    display(tabla_curso[tabla_curso["Bajo_rendimiento"] > 0].head(20))

    tabla_asig = _tabla_agrupada(df, ["Nombre_asignatura", "Area"])
    print(f"\nPor asignatura:")
    display(tabla_asig[tabla_asig["Bajo_rendimiento"] > 0].head(20))

    tabla_area = _tabla_agrupada(df, ["Area"])
    print(f"\nPor área:")
    display(tabla_area[tabla_area["Bajo_rendimiento"] > 0])

    revision = df[df["condicion_de_alerta"]][
        ["Nombre_completo_estudiante", "Sede", "Nombre_programa_limpio", "Grupo",
         "Primer Seguimiento", "Segundo Seguimiento", "Tercer Seguimiento", col_nota]
    ].drop_duplicates()
    print(f"\nEstudiantes en revisión ({titulo}):")
    display(revision)
